In [1]:
import time
import torch
import logging
import warnings
from torch.profiler import DeviceType, ProfilerActivity, profile
from transformers import AutoModelForCausalLM

warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch._inductor").setLevel(logging.ERROR)

MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"
DEVICE = "cuda"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map=DEVICE)

input_prefill = torch.randint(0, model.config.vocab_size, (1, 1000), device=DEVICE)
input_decode = torch.randint(0, model.config.vocab_size, (1, 1), device=DEVICE)


def benchmark(test_model, input_tensor, name):
    print(f"Warmup {name}")
    for _ in range(3):
        with torch.no_grad():
            test_model(input_tensor)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(5):
        with torch.no_grad():
            test_model(input_tensor)
    torch.cuda.synchronize()
    avg_time_ms = ((time.perf_counter() - start) / 5) * 1000

    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        with torch.no_grad():
            test_model(input_tensor)

    gpu_events = [e for e in prof.events() if e.device_type == DeviceType.CUDA]

    print(f"Results for {name}:")
    print(f"Latency: {avg_time_ms:.2f} ms")
    print(f"Kernel count: {len(gpu_events)}\n")


def measure_modes(test_model, name):
    print(f"Warming up {name}")
    for _ in range(3):
        with torch.no_grad():
            test_model(input_prefill)
            test_model(input_decode)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(5):
        with torch.no_grad():
            test_model(input_prefill)
    torch.cuda.synchronize()
    prefill_time = ((time.perf_counter() - start) / 5) * 1000

    start = time.perf_counter()
    for _ in range(20):
        with torch.no_grad():
            test_model(input_decode)
    torch.cuda.synchronize()
    decode_time = ((time.perf_counter() - start) / 20) * 1000

    print(f"Prefill latency : {prefill_time:.2f} ms")
    print(f"Decode latency  : {decode_time:.2f} ms\n")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [2]:
print("--- EAGER MODE ---")
benchmark(model, input_prefill, "Prefill (1000 tokens)")
benchmark(model, input_decode, "Decode (1 token)")

--- EAGER MODE ---
Warmup Prefill (1000 tokens)
Results for Prefill (1000 tokens):
Latency: 141.48 ms
Kernel count: 2289

Warmup Decode (1 token)
Results for Decode (1 token):
Latency: 22.11 ms
Kernel count: 2037



In [3]:
print("--- COMPILED MODE ---")
compiled_base = torch.compile(model)
benchmark(compiled_base, input_prefill, "Prefill (1000 tokens)")
benchmark(compiled_base, input_decode, "Decode (1 token)")

--- COMPILED MODE ---
Warmup Prefill (1000 tokens)
Results for Prefill (1000 tokens):
Latency: 139.60 ms
Kernel count: 976

Warmup Decode (1 token)
Results for Decode (1 token):
Latency: 19.93 ms
Kernel count: 616



In [4]:
print("--- EXPORTING PROFILE TRACE ---")
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof_trace:
    with torch.no_grad():
        compiled_base(input_decode)
prof_trace.export_chrome_trace("compiled_trace.json")
print("Trace saved to: compiled_trace.json")

--- EXPORTING PROFILE TRACE ---
Trace saved to: compiled_trace.json


In [7]:
print("--- BENCHMARK COMPILATION MODES ---")
measure_modes(compiled_base, "Standard torch.compile")

print("Compilation: reduce-overhead (CUDA Graphs)")
compiled_graphs = torch.compile(model, mode="reduce-overhead")
measure_modes(compiled_graphs, "Mode reduce-overhead")

print("Compilation: max-autotune")
compiled_auto = torch.compile(model, mode="max-autotune")
measure_modes(compiled_auto, "Mode max-autotune")

--- BENCHMARK COMPILATION MODES ---
Warming up Standard torch.compile
Prefill latency : 138.94 ms
Decode latency  : 19.45 ms

Compilation: reduce-overhead (CUDA Graphs)
Warming up Mode reduce-overhead
Prefill latency : 142.21 ms
Decode latency  : 19.13 ms

Compilation: max-autotune
Warming up Mode max-autotune
Prefill latency : 142.73 ms
Decode latency  : 19.01 ms

